# 03 - Découpage chronologique et préparation sans fuite

.

## 1. Imports et dossier du projet

Nous réutilisons les fonctions de chargement et le pipeline déjà employés dans `02`. Les chemins sont calculés à partir du projet, sans nom d'utilisateur inscrit dans le notebook.

In [1]:
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "ml" / "src").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Ouvre le notebook depuis le dossier du projet.")
    PROJECT_ROOT = PROJECT_ROOT.parent

ML_SRC = PROJECT_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

from data.load_data import (
    load_development_dataset,
    split_chronologically,
    summarize_splits,
)
from preprocessing.pipeline import build_preprocessing_pipeline, split_features_target

print("Racine du projet :", PROJECT_ROOT)

Racine du projet : C:\Users\Lenovo\Documents\fintech-mlops-lsi


## 2. Charger le fichier de développement

Le fichier doit être un vrai CSV dans `data/raw/fraudTrain.csv`. Nous contrôlons les dates, la cible binaire et les identifiants utilisés pour détecter les chevauchements. Aucune ligne brute contenant des données personnelles n'est affichée.

In [2]:
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "fraudTrain.csv"
print(TRAIN_PATH)
print("Existe :", TRAIN_PATH.exists())
if not TRAIN_PATH.is_file():
    raise FileNotFoundError("Place le vrai fichier fraudTrain.csv dans data/raw/.")

df, train_summary = load_development_dataset(TRAIN_PATH)
assert df.index.is_unique, "L'index source doit être unique."
assert df["is_fraud"].notna().all(), "Cible manquante."
assert df["is_fraud"].isin([0, 1]).all(), "La cible doit contenir uniquement 0 ou 1."
assert "trans_num" in df.columns, "Identifiant trans_num absent."
assert df["trans_num"].notna().all(), "Identifiant de transaction manquant."
assert df["trans_num"].is_unique, "Des identifiants de transaction sont dupliqués."

source_dates = pd.to_datetime(
    df["trans_date_trans_time"], format="%Y-%m-%d %H:%M:%S", errors="raise"
)
assert source_dates.notna().all(), "Date de transaction manquante."
assert source_dates.dt.tz is None, "Les dates doivent être locales et naïves."
source_is_sorted = source_dates.is_monotonic_increasing
print("Dimensions du fichier :", df.shape)
print("Déjà trié chronologiquement :", source_is_sorted)
if not source_is_sorted:
    print("Le découpage utilisera une copie triée en mémoire, sans modifier le CSV.")

C:\Users\Lenovo\Documents\fintech-mlops-lsi\data\raw\fraudTrain.csv
Existe : True
Dimensions du fichier : (1296675, 23)
Déjà trié chronologiquement : False
Le découpage utilisera une copie triée en mémoire, sans modifier le CSV.


Dimensions du fichier : (1296675, 23)
Déjà trié chronologiquement : False
Le découpage utilisera une copie triée en mémoire, sans modifier le CSV.


## 3. Créer les trois zones

La fonction existante trie de manière stable si nécessaire, puis découpe par nombre de lignes dans l'ordre du temps. Les deux premiers effectifs sont arrondis à l'entier inférieur ; la dernière zone reçoit le reste. Le total est donc conservé, avec des proportions proches de 70/15/15 à une ligne près.

In [4]:
DEV_RATIO = 0.70
PROBA_RATIO = 0.15
CONFORMAL_RATIO = 0.15

splits = split_chronologically(
    df,
    dev_ratio=DEV_RATIO,
    proba_calibration_ratio=PROBA_RATIO,
    conformal_calibration_ratio=CONFORMAL_RATIO,
    sort_if_needed=True,
)
df_dev = splits["development"]
df_proba = splits["probability_calibration"]
df_conformal = splits["conformal_calibration"]

total_rows = len(df)
expected_dev = int(total_rows * DEV_RATIO)
expected_proba = int(total_rows * PROBA_RATIO)
expected_conformal = total_rows - expected_dev - expected_proba

assert len(df_dev) == expected_dev, "Taille development incorrecte."
assert len(df_proba) == expected_proba, "Taille calibration probabilités incorrecte."
assert len(df_conformal) == expected_conformal, "Taille calibration conforme incorrecte."
assert all(len(zone) > 0 for zone in splits.values()), "Une zone est vide."
assert sum(len(zone) for zone in splits.values()) == total_rows, "Des lignes ont été perdues."
print("Tailles et conservation des lignes : OK")

Tailles et conservation des lignes : OK


## 4. Afficher les dates, effectifs et taux de fraude

Ce tableau est calculé sur les données réellement chargées. Les taux de fraude peuvent différer entre périodes : nous les décrivons sans modifier le découpage ni équilibrer les calibrations. Les taux et pourcentages sont distingués explicitement.

In [5]:
split_summary = summarize_splits(splits)
split_summary["part_du_fichier_pct"] = 100 * split_summary["rows"] / total_rows
split_summary["taux_fraude_pct"] = 100 * split_summary["fraud_rate"]
display(split_summary)
print(split_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

,zone,start_date,end_date,rows,fraud_count,fraud_rate,part_du_fichier_pct,taux_fraude_pct
0,development,2019-01-01 00:00:18,2019-12-28 17:18:24,907672,5121,0.005642,69.999961,0.564191
1,probability_calibration,2019-12-28 17:18:38,2020-04-03 17:53:52,194501,1252,0.006437,14.999981,0.643698
2,conformal_calibration,2020-04-03 17:54:44,2020-06-21 12:13:37,194502,1133,0.005825,15.000058,0.582513


                   zone          start_date            end_date   rows  fraud_count  fraud_rate  part_du_fichier_pct  taux_fraude_pct
            development 2019-01-01 00:00:18 2019-12-28 17:18:24 907672         5121    0.005642            69.999961         0.564191
probability_calibration 2019-12-28 17:18:38 2020-04-03 17:53:52 194501         1252    0.006437            14.999981         0.643698
  conformal_calibration 2020-04-03 17:54:44 2020-06-21 12:13:37 194502         1133    0.005825            15.000058         0.582513


## 5. Vérifier l'ordre chronologique

Chaque zone doit être triée. La fin du développement doit précéder strictement le début de la calibration des probabilités, qui doit précéder la calibration conforme. Si des transactions partagent la même date aux frontières, ce contrôle arrête le notebook pour examiner le cas ; il ne change pas automatiquement le découpage.

In [6]:
zone_dates = {}
for name, zone in splits.items():
    dates = pd.to_datetime(
        zone["trans_date_trans_time"], format="%Y-%m-%d %H:%M:%S", errors="raise"
    )
    assert dates.notna().all(), f"Date manquante dans {name}."
    assert dates.is_monotonic_increasing, f"Zone non triée : {name}."
    zone_dates[name] = dates
    print(name, ": ordre chronologique OK")

assert zone_dates["development"].max() < zone_dates["probability_calibration"].min(), (
    "Dates communes ou ordre incorrect à la première frontière."
)
assert zone_dates["probability_calibration"].max() < zone_dates["conformal_calibration"].min(), (
    "Dates communes ou ordre incorrect à la seconde frontière."
)
print("Frontières temporelles strictement ordonnées : OK")

development : ordre chronologique OK
probability_calibration : ordre chronologique OK
conformal_calibration : ordre chronologique OK
Frontières temporelles strictement ordonnées : OK


probability_calibration : ordre chronologique OK


conformal_calibration : ordre chronologique OK
Frontières temporelles strictement ordonnées : OK


## 6. Vérifier l'absence de chevauchement

Nous comparons les index source et les identifiants `trans_num` de chaque paire de zones. Aucun identifiant de transaction n'est affiché. L'union des index doit retrouver exactement toutes les lignes source. Cela garantit des transactions disjointes ; un même client peut naturellement réaliser plusieurs transactions à des dates différentes.

In [7]:
overlap_rows = []
for left, right in combinations(splits, 2):
    index_overlap = splits[left].index.intersection(splits[right].index)
    transaction_overlap = pd.Index(splits[left]["trans_num"]).intersection(
        pd.Index(splits[right]["trans_num"])
    )
    assert len(index_overlap) == 0, f"Index partagés entre {left} et {right}."
    assert len(transaction_overlap) == 0, f"Transactions partagées entre {left} et {right}."
    overlap_rows.append({
        "zone_1": left, "zone_2": right,
        "index_communs": len(index_overlap),
        "transactions_communes": len(transaction_overlap),
    })

all_split_index = df_dev.index.append(df_proba.index).append(df_conformal.index)
assert all_split_index.is_unique, "Une ligne figure dans plusieurs zones."
assert len(all_split_index.difference(df.index)) == 0, "Index inconnu."
assert len(df.index.difference(all_split_index)) == 0, "Ligne source absente."
display(pd.DataFrame(overlap_rows))
print("Chevauchements : 0 ; toutes les lignes source sont présentes.")

,zone_1,zone_2,index_communs,transactions_communes
0,development,probability_calibration,0,0
1,development,conformal_calibration,0,0
2,probability_calibration,conformal_calibration,0,0


Chevauchements : 0 ; toutes les lignes source sont présentes.


## 7. Séparer les entrées et la cible

Nous gardons les données brutes de chaque zone et séparons `X` (entrées) et `y` (cible). Les calibrations restent à leur distribution naturelle. Leurs étiquettes ne servent pas à ajuster le preprocessing.

In [8]:
X_dev, y_dev = split_features_target(df_dev)
X_proba, y_proba = split_features_target(df_proba)
X_conformal, y_conformal = split_features_target(df_conformal)

for name, X, y in [
    ("development", X_dev, y_dev),
    ("probability_calibration", X_proba, y_proba),
    ("conformal_calibration", X_conformal, y_conformal),
]:
    assert "is_fraud" not in X.columns, f"La cible est présente dans X : {name}."
    assert X.index.equals(y.index), f"X et y sont désalignés : {name}."
    print(name, ": X", X.shape, "; y", y.shape)

development : X (907672, 22) ; y (907672,)
probability_calibration : X (194501, 22) ; y (194501,)
conformal_calibration : X (194502, 22) ; y (194502,)


## 8. Ajuster le preprocessing sur le développement seulement

Un seul appel à `fit` est effectué, sur `X_dev`. Le scaler apprend ses paramètres sur le développement et l'encodeur apprend uniquement ses modalités. Cet ajustement global prépare la démonstration : lors du benchmark, un pipeline neuf sera réajusté dans chaque pli d'entraînement, sans voir le pli de validation.

In [9]:
preprocessing_pipeline = build_preprocessing_pipeline()
preprocessing_pipeline.fit(X_dev)

columns_step = preprocessing_pipeline.named_steps["columns"]
scaler = columns_step.named_transformers_["numeric"]
encoder = columns_step.named_transformers_["categorical"]

assert np.all(np.asarray(scaler.n_samples_seen_) == len(X_dev)), (
    "Le scaler n'a pas vu uniquement l'effectif development attendu."
)
for column, learned_categories in zip(["gender", "category", "state"], encoder.categories_):
    expected_categories = set(X_dev[column].astype(str).unique())
    assert set(learned_categories) == expected_categories, f"Modalités inattendues : {column}."

feature_names = preprocessing_pipeline.get_feature_names_out()
print("Lignes utilisées pour ajuster le preprocessing :", int(scaler.n_samples_seen_))
print("Zone utilisée : development")
print("Nombre de features de sortie :", len(feature_names))
print("Scaler et modalités de l'encodeur vérifiés sur development : OK")

Lignes utilisées pour ajuster le preprocessing : 907672
Zone utilisée : development
Nombre de features de sortie : 77
Scaler et modalités de l'encodeur vérifiés sur development : OK


## 9. Transformer sans réajuster : vérification légère

Nous transformons seulement cinq lignes par zone pour vérifier le fonctionnement et limiter la mémoire. Il s'agit d'un contrôle de forme, pas d'une calibration. Les valeurs apprises du scaler et de l'encodeur doivent rester identiques après ces appels à `transform`. Aucune matrice prétraitée complète n'est exportée.

In [11]:
mean_before = scaler.mean_.copy()
scale_before = scaler.scale_.copy()
count_before = np.asarray(scaler.n_samples_seen_).copy()
categories_before = [values.copy() for values in encoder.categories_]

for name, X in [
    ("development", X_dev),
    ("probability_calibration", X_proba),
    ("conformal_calibration", X_conformal),
]:
    preview = preprocessing_pipeline.transform(X.iloc[:5])
    assert preview.shape == (min(5, len(X)), len(feature_names)), f"Dimensions incorrectes : {name}."
    assert np.isfinite(preview).all(), f"Valeurs non finies : {name}."
    print(name, ": transform seulement", preview.shape, "; valeurs finies : OK")

assert np.array_equal(scaler.mean_, mean_before), "Moyennes du scaler modifiées."
assert np.array_equal(scaler.scale_, scale_before), "Échelles du scaler modifiées."
assert np.array_equal(np.asarray(scaler.n_samples_seen_), count_before), "Effectif du scaler modifié."
for before, after in zip(categories_before, encoder.categories_):
    assert np.array_equal(before, after), "Modalités de l'encodeur modifiées."
print("Paramètres appris inchangés après transformation des trois zones : OK")

development : transform seulement (5, 77) ; valeurs finies : OK
probability_calibration : transform seulement (5, 77) ; valeurs finies : OK
conformal_calibration : transform seulement (5, 77) ; valeurs finies : OK
Paramètres appris inchangés après transformation des trois zones : OK


conformal_calibration : transform seulement (5, 77) ; valeurs finies : OK
Paramètres appris inchangés après transformation des trois zones : OK
